# EX: Probabilistic Inference by Enumeration

In multi-domain command and control (C2), tactical commanders frequently face incomplete operational pictures. Sensors report symptoms (e.g., radar noise, communication timeouts, missile aborts), but the underlying operational drivers (e.g., electronic attack, adversary decoys, adverse micro-climates) cannot be directly observed.

Probabilistic Inference allows an AI decision-support system to condition on observed evidence and mathematically infer the probability distribution of unobserved query variables by systematically summing out hidden variables.

In this exercise, you will programmatically model the theater air-strike planning network from CS471:

$$W \to V, \quad W \to C, \quad V \to M, \quad C \to M$$

Where:

$W \in \{\text{Clear}, \text{Severe}\}$: Sector Weather

$V \in \{\text{Good}, \text{Poor}\}$: Visual Target Acquisition

$C \in \{\text{Reliable}, \text{Disrupted}\}$: SATCOM Link

$M \in \{\text{Success}, \text{Abort}\}$: Strike Mission Outcome

You will implement an exact Inference by Enumeration algorithm that partitions variables into Query, Evidence, and Hidden sets, computes unnormalized joint probabilities, applies the normalization constant $\alpha$, and outputs tactical posterior distributions.

**Lab Workflow & Steps Taken in the Code**

1. Graph Representation & CPT Structure

Represent the local conditional probability tables for $P(W)$, $P(V \mid W)$, $P(C \mid W)$, and $P(M \mid V, C)$ as structured nested Python dictionaries.

2. Joint Factorization Primitive

Implement a core helper function compute_joint(w, v, c, m) that evaluates atomic configurations via the Bayesian network chain rule:


$$P(W, V, C, M) = P(W) \cdot P(V \mid W) \cdot P(C \mid W) \cdot P(M \mid V, C)$$

3. General Inference by Enumeration Engine

Construct a general query function infer_enumeration(query_var, evidence, all_vars, domains) that:

Identifies the hidden variables to sum out: $H = \text{AllVars} \setminus (\{Q\} \cup \mathbf{E})$.

Iterates through every state of the query variable $Q$.

Sums joint probabilities over all combinatorial assignments of $H$.

Normalizes the resulting distribution vector using $\alpha = \frac{1}{\sum_q P(Q=q, \mathbf{E})}$.

4. Operational Case Studies

**Case Study A (Causal / Forward Prediction)**: Compute $P(M \mid W = \text{Severe})$ when weather is known but sensors and communications are unobserved.

**Case Study B (Diagnostic / Root-Cause Diagnosis)**: Compute $P(W \mid M = \text{Abort}, C = \text{Reliable})$ to diagnose whether bad weather was responsible for an abort when communications remained operational.



In [1]:
import itertools
import pandas as pd

# -------------------------------------------------------------------------
# Step 1: Define Conditional Probability Tables (CPTs)
# -------------------------------------------------------------------------
p_W = {
    'Clear': 0.80, 
    'Severe': 0.20
}

p_V_given_W = {
    'Clear':  {'Good': 0.90, 'Poor': 0.10},
    'Severe': {'Good': 0.30, 'Poor': 0.70}
}

p_C_given_W = {
    'Clear':  {'Reliable': 0.85, 'Disrupted': 0.15},
    'Severe': {'Reliable': 0.40, 'Disrupted': 0.60}
}

p_M_given_VC = {
    ('Good', 'Reliable'):   {'Success': 0.95, 'Abort': 0.05},
    ('Good', 'Disrupted'):  {'Success': 0.60, 'Abort': 0.40},
    ('Poor', 'Reliable'):   {'Success': 0.50, 'Abort': 0.50},
    ('Poor', 'Disrupted'):  {'Success': 0.10, 'Abort': 0.90}
}

# Variable domains dictionary
variable_domains = {
    'W': ['Clear', 'Severe'],
    'V': ['Good', 'Poor'],
    'C': ['Reliable', 'Disrupted'],
    'M': ['Success', 'Abort']
}

# -------------------------------------------------------------------------
# Step 2: Joint Factorization Function
# -------------------------------------------------------------------------
def compute_joint(assignment):
    """
    Evaluates P(W, V, C, M) for a complete variable assignment dictionary.
    """
    w = assignment['W']
    v = assignment['V']
    c = assignment['C']
    m = assignment['M']
    
    return p_W[w] * p_V_given_W[w][v] * p_C_given_W[w][c] * p_M_given_VC[(v, c)][m]

# -------------------------------------------------------------------------
# Step 3: Exact Inference by Enumeration Engine
# -------------------------------------------------------------------------
def infer_enumeration(query_var, evidence, all_domains):
    """
    Computes P(QueryVar | Evidence) via exact marginalization over hidden variables.
    """
    hidden_vars = [v for v in all_domains.keys() if v != query_var and v not in evidence]
    hidden_domains = [all_domains[v] for v in hidden_vars]
    
    unnormalized_probs = {}
    
    # Iterate over all possible states of the query variable
    for q_val in all_domains[query_var]:
        total_prob_mass = 0.0
        
        # Marginalize over all possible states of hidden variables
        if hidden_domains:
            for h_combination in itertools.product(*hidden_domains):
                assignment = dict(evidence)
                assignment[query_var] = q_val
                for h_var, h_val in zip(hidden_vars, h_combination):
                    assignment[h_var] = h_val
                
                total_prob_mass += compute_joint(assignment)
        else:
            assignment = dict(evidence)
            assignment[query_var] = q_val
            total_prob_mass = compute_joint(assignment)
            
        unnormalized_probs[q_val] = total_prob_mass
        
    # Normalization step: alpha = 1 / sum(unnormalized_probs)
    normalization_constant_alpha = 1.0 / sum(unnormalized_probs.values())
    normalized_distribution = {
        q_val: prob * normalization_constant_alpha 
        for q_val, prob in unnormalized_probs.items()
    }
    
    return normalized_distribution, unnormalized_probs, normalization_constant_alpha

# -------------------------------------------------------------------------
# Step 4: Execute Operational Case Studies
# -------------------------------------------------------------------------
print("=" * 65)
print("TACTICAL BAYESIAN INFERENCE ENGINE (CS471 LAB 21)")
print("=" * 65 + "\n")

# Case Study A: Causal Forward Prediction
# Query: P(Mission | Weather = Severe)
evidence_a = {'W': 'Severe'}
post_m_given_sev, unnorm_a, alpha_a = infer_enumeration('M', evidence_a, variable_domains)

print("--- CASE STUDY A: CAUSAL INFERENCE ---")
print(f"Query: P(Mission | Weather = Severe)")
print(f"Evidence: {evidence_a}")
for outcome, prob in post_m_given_sev.items():
    print(f"  P(M = {outcome} | W = Severe): {prob:.4f} ({prob*100:.2f}%)")
print()

# Case Study B: Diagnostic Root-Cause Inference
# Query: P(Weather | Mission = Abort, SATCOM = Reliable)
evidence_b = {'M': 'Abort', 'C': 'Reliable'}
post_w_given_diag, unnorm_b, alpha_b = infer_enumeration('W', evidence_b, variable_domains)

print("--- CASE STUDY B: DIAGNOSTIC INFERENCE ---")
print(f"Query: P(Weather | Mission = Abort, SATCOM = Reliable)")
print(f"Evidence: {evidence_b}")
print(f"Unnormalized Weights: {unnorm_b}")
print(f"Normalization Constant (alpha): {alpha_b:.4f}")
for weather_state, prob in post_w_given_diag.items():
    print(f"  P(W = {weather_state} | Abort, Reliable): {prob:.4f} ({prob*100:.2f}%)")
print("\n" + "=" * 65)


TACTICAL BAYESIAN INFERENCE ENGINE (CS471 LAB 21)

--- CASE STUDY A: CAUSAL INFERENCE ---
Query: P(Mission | Weather = Severe)
Evidence: {'W': 'Severe'}
  P(M = Success | W = Severe): 0.4040 (40.40%)
  P(M = Abort | W = Severe): 0.5960 (59.60%)

--- CASE STUDY B: DIAGNOSTIC INFERENCE ---
Query: P(Weather | Mission = Abort, SATCOM = Reliable)
Evidence: {'M': 'Abort', 'C': 'Reliable'}
Unnormalized Weights: {'Clear': 0.0646, 'Severe': 0.029199999999999997}
Normalization Constant (alpha): 10.6610
  P(W = Clear | Abort, Reliable): 0.6887 (68.87%)
  P(W = Severe | Abort, Reliable): 0.3113 (31.13%)



## Interpreting the Results

**Causal Degradation (Case Study A)**

Under baseline conditions with clear weather, mission success exceeds $85\%$. However, when evidence of Severe Weather is introduced ($W = \text{Severe}$), marginalizing over the degraded visual and communication states yields:


$$P(M = \text{Success} \mid W = \text{Severe}) = 0.4040 \quad (40.40\%)$$


Severe weather reduces mission success probability by more than half, informing commanders to consider holding reserve strikes.

**Diagnostic Diagnosis & Explaining Away (Case Study B)**

When an autonomous strike package aborts, intelligence officers must determine if adverse weather was the primary cause.

Baseline Weather Prior: $P(W = \text{Severe}) = 0.2000$ ($20.0\%$).

Posterior After Abort & Reliable SATCOM: $P(W = \text{Severe} \mid M = \text{Abort}, C = \text{Reliable}) = 0.3113$ ($31.13\%$).

Even though a mission failure occurred, the posterior probability of severe weather is only $31.13\%$, meaning that Clear Weather remains the dominant hypothesis ($68.87\%$). Because the SATCOM link remained Reliable, the system recognizes that communications did not experience weather-related disruption, partially "explaining away" the hypothesis of a catastrophic weather event.

**Mathematical Consistency**

In both case studies, the sum of normalized posterior probabilities equals exactly $1.0000$, validating that the enumeration engine accounts for all probability mass across the variable domains without leakage.# Bayesian Inference Python Lab